In [6]:
# Install required packages
!pip install torch torchaudio librosa numpy matplotlib scipy gtts requests soundfile


   ---------------------------------------- 0/2 [click]
   ---------------------------------------- 0/2 [click]
   ---------------------------------------- 0/2 [click]
   -------------------- ------------------- 1/2 [gtts]
   -------------------- ------------------- 1/2 [gtts]
   -------------------- ------------------- 1/2 [gtts]
   -------------------- ------------------- 1/2 [gtts]
   ---------------------------------------- 2/2 [gtts]



In [7]:
import torch
import torch.nn as nn
import torchaudio
import librosa
import numpy as np
import matplotlib.pyplot as plt
import os
import urllib.request
import zipfile
from scipy import signal
import soundfile as sf
from pathlib import Path

print("✅ All packages imported successfully")

✅ All packages imported successfully


In [9]:
def download_acoustic_dataset():
    """Download impulse responses for different acoustic environments"""
    print("📥 Downloading acoustic environment dataset...")
    
    os.makedirs('impulse_responses', exist_ok=True)
    os.makedirs('content_audio', exist_ok=True)
    
    # Download MIT McDermott Impulse Response Dataset
    print("Downloading impulse responses...")
    impulse_urls = {
        'small_room': 'https://mcdermottlab.mit.edu/Reverb_dataset_SIG/IRs/MIT_KEMAR_normal_pinna/1_A_softlivingroom.wav',
        'large_hall': 'https://mcdermottlab.mit.edu/Reverb_dataset_SIG/IRs/MIT_KEMAR_normal_pinna/1_A_largehall.wav',
        'church': 'https://mcdermottlab.mit.edu/Reverb_dataset_SIG/IRs/MIT_KEMAR_normal_pinna/1_A_church.wav',
        'stairwell': 'https://mcdermottlab.mit.edu/Reverb_dataset_SIG/IRs/MIT_KEMAR_normal_pinna/1_A_stairwell.wav'
    }
    
    for env_name, url in impulse_urls.items():
        try:
            file_path = f'impulse_responses/{env_name}.wav'
            urllib.request.urlretrieve(url, file_path)
            print(f"✅ Downloaded: {env_name}")
        except:
            print(f"❌ Failed to download {env_name}, creating synthetic IR")
            create_synthetic_impulse(env_name)
    
    # Create some test content audio
    create_test_audio()
    
    print("✅ Dataset preparation complete!")

def create_synthetic_impulse(env_name):
    """Create synthetic impulse responses if downloads fail"""
    duration = 2.0  # seconds
    sr = 22050
    t = np.linspace(0, duration, int(sr * duration))
    
    if env_name == 'small_room':
        # Short reverb, quick decay
        impulse = np.exp(-8 * t) * np.sin(2 * np.pi * 100 * t)
    elif env_name == 'large_hall':
        # Long reverb, slow decay
        impulse = np.exp(-2 * t) * np.sin(2 * np.pi * 80 * t)
    elif env_name == 'church':
        # Very long reverb with modulation
        impulse = np.exp(-1.5 * t) * np.sin(2 * np.pi * 60 * t) * (1 + 0.3 * np.sin(2 * np.pi * 5 * t))
    else:  # stairwell
        # Metallic, ringing reverb
        impulse = np.exp(-4 * t) * np.sin(2 * np.pi * 200 * t) * (1 + 0.5 * np.sin(2 * np.pi * 10 * t))
    
    # Normalize
    impulse = impulse / np.max(np.abs(impulse))
    sf.write(f'impulse_responses/{env_name}.wav', impulse, sr)
    print(f"✅ Created synthetic: {env_name}")

def create_test_audio():
    """Create test audio content"""
    from gtts import gTTS
    
    test_sentences = [
        "This is a test recording in a small room",
        "The quick brown fox jumps over the lazy dog",
        "Audio acoustic transformation is fascinating",
        "This voice will sound like different environments"
    ]
    
    for i, sentence in enumerate(test_sentences):
        tts = gTTS(text=sentence, lang='en', slow=False)
        tts.save(f'content_audio/test_{i}.wav')
    
    # Also create a simple sine wave test tone
    sr = 22050
    duration = 3.0
    t = np.linspace(0, duration, int(sr * duration))
    tone = 0.5 * np.sin(2 * np.pi * 440 * t)  # A440 sine wave
    sf.write('content_audio/test_tone.wav', tone, sr)
    
    print("✅ Created test audio content")

# Download the dataset
download_acoustic_dataset()

📥 Downloading acoustic environment dataset...
❌ Failed to download small_room, creating synthetic IR
✅ Created synthetic: small_room
❌ Failed to download large_hall, creating synthetic IR
✅ Created synthetic: large_hall
❌ Failed to download church, creating synthetic IR
✅ Created synthetic: church
❌ Failed to download stairwell, creating synthetic IR
✅ Created synthetic: stairwell
✅ Created test audio content
✅ Dataset preparation complete!


In [10]:
class AcousticTransfer:
    def __init__(self):
        self.sr = 22050
        
    def load_audio(self, file_path):
        """Load audio file and convert to mono"""
        audio, sr = librosa.load(file_path, sr=self.sr)
        return audio, sr
    
    def apply_convolution_reverb(self, dry_audio, impulse_response):
        """Apply convolution reverb to dry audio"""
        # Convolve dry audio with impulse response
        wet_audio = signal.convolve(dry_audio, impulse_response, mode='full')
        
        # Normalize to prevent clipping
        wet_audio = wet_audio / np.max(np.abs(wet_audio))
        
        return wet_audio
    
    def spectral_transfer(self, dry_audio, target_env_audio, mix_ratio=0.7):
        """Transfer spectral characteristics using FFT"""
        # Compute FFT of both signals
        dry_fft = np.fft.rfft(dry_audio)
        target_fft = np.fft.rfft(target_env_audio)
        
        # Get magnitudes and phases
        dry_mag = np.abs(dry_fft)
        dry_phase = np.angle(dry_fft)
        target_mag = np.abs(target_fft)
        
        # Blend magnitudes
        blended_mag = (1 - mix_ratio) * dry_mag + mix_ratio * target_mag
        
        # Reconstruct signal
        transformed_fft = blended_mag * np.exp(1j * dry_phase)
        transformed_audio = np.fft.irfft(transformed_fft)
        
        return transformed_audio
    
    def analyze_acoustic_features(self, audio):
        """Analyze acoustic features of audio"""
        # Compute reverb time (RT60 approximation)
        energy = np.cumsum(audio[::-1]**2)[::-1]
        rt60_threshold = energy[0] * 0.001  # -60 dB
        rt60_samples = np.argmax(energy < rt60_threshold)
        rt60 = rt60_samples / self.sr
        
        # Spectral centroid (brightness)
        spectral_centroid = librosa.feature.spectral_centroid(y=audio, sr=self.sr)[0]
        brightness = np.mean(spectral_centroid)
        
        return {
            'reverb_time': rt60,
            'brightness': brightness,
            'length': len(audio) / self.sr
        }

def demonstrate_acoustic_transfer():
    """Demonstrate the acoustic transfer on test audio"""
    transfer = AcousticTransfer()
    
    # Load test content
    content_audio, _ = transfer.load_audio('content_audio/test_0.wav')
    
    print("🏠 Testing acoustic environment transfer...")
    
    # Get all impulse responses
    impulse_files = list(Path('impulse_responses').glob('*.wav'))
    
    results = {}
    for impulse_file in impulse_files:
        env_name = impulse_file.stem
        print(f"🎯 Processing: {env_name}")
        
        # Load impulse response
        impulse, _ = transfer.load_audio(str(impulse_file))
        
        # Apply convolution reverb
        transformed_audio = transfer.apply_convolution_reverb(content_audio, impulse)
        
        # Save result
        output_path = f'output_{env_name}.wav'
        sf.write(output_path, transformed_audio, transfer.sr)
        
        # Analyze features
        original_features = transfer.analyze_acoustic_features(content_audio)
        transformed_features = transfer.analyze_acoustic_features(transformed_audio)
        
        results[env_name] = {
            'audio': transformed_audio,
            'original_features': original_features,
            'transformed_features': transformed_features,
            'file_path': output_path
        }
        
        print(f"  Original RT60: {original_features['reverb_time']:.2f}s")
        print(f"  Transformed RT60: {transformed_features['reverb_time']:.2f}s")
        print(f"  Saved: {output_path}")
    
    return results

# Run the demonstration
results = demonstrate_acoustic_transfer()

🏠 Testing acoustic environment transfer...
🎯 Processing: church
  Original RT60: 2.91s
  Transformed RT60: 4.19s
  Saved: output_church.wav
🎯 Processing: large_hall
  Original RT60: 2.91s
  Transformed RT60: 3.87s
  Saved: output_large_hall.wav
🎯 Processing: small_room
  Original RT60: 2.91s
  Transformed RT60: 3.34s
  Saved: output_small_room.wav
🎯 Processing: stairwell
  Original RT60: 2.91s
  Transformed RT60: 3.21s
  Saved: output_stairwell.wav


In [11]:
class AcousticTransferNN(nn.Module):
    """Neural network for learning acoustic transformations"""
    def __init__(self):
        super().__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 64, 15, padding=7),
            nn.ReLU(),
            nn.Conv1d(64, 128, 15, padding=7),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(256)  # Fixed size encoding
        )
        
        # Style transformation
        self.transform = nn.Sequential(
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU()
        )
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(256, 128, 15, padding=7),
            nn.ReLU(),
            nn.ConvTranspose1d(128, 64, 15, padding=7),
            nn.ReLU(),
            nn.ConvTranspose1d(64, 1, 15, padding=7),
            nn.Tanh()
        )
    
    def forward(self, x, style_code):
        # Encode
        encoded = self.encoder(x)
        
        # Apply style transformation
        batch, channels, length = encoded.shape
        encoded_flat = encoded.view(batch, -1)
        transformed = self.transform(encoded_flat)
        transformed = transformed.view(batch, channels, length)
        
        # Decode
        output = self.decoder(transformed)
        return output

def train_acoustic_model():
    """Train the neural network on acoustic transformations"""
    print("🧠 Training acoustic transfer model...")
    
    model = AcousticTransferNN()
    
    # For demo purposes, we'll create synthetic training data
    # In a real project, you'd use actual impulse response pairs
    
    # Generate training samples
    def create_training_sample():
        sr = 22050
        duration = 1.0
        t = np.linspace(0, duration, int(sr * duration))
        
        # Dry signal (simple tone + noise)
        dry = 0.3 * np.sin(2 * np.pi * 440 * t) + 0.1 * np.random.randn(len(t))
        
        # Wet signal (with synthetic reverb)
        wet = dry.copy()
        for i in range(len(wet)):
            if i > 100:  # Add delayed reflections
                wet[i] += 0.5 * wet[i-100]
            if i > 200:
                wet[i] += 0.3 * wet[i-200]
        
        return torch.FloatTensor(dry).unsqueeze(0).unsqueeze(0), torch.FloatTensor(wet).unsqueeze(0).unsqueeze(0)
    
    # Simple training loop
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    
    print("Training model (this may take a moment)...")
    for epoch in range(100):
        total_loss = 0
        for _ in range(10):  # 10 batches per epoch
            dry, wet = create_training_sample()
            style_code = torch.randn(1, 256)  # Random style code for demo
            
            optimizer.zero_grad()
            output = model(dry, style_code)
            loss = criterion(output, wet)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        if epoch % 20 == 0:
            print(f"Epoch {epoch}, Loss: {total_loss/10:.4f}")
    
    print("✅ Model training complete (demo version)")
    return model

# Train the model
model = train_acoustic_model()

🧠 Training acoustic transfer model...
Training model (this may take a moment)...


RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x32768 and 256x512)

In [ ]:
def run_complete_demo():
    """Run the complete acoustic transfer demo"""
    print("🎵 COMPLETE ACOUSTIC TRANSFER DEMO")
    print("=" * 50)
    
    # Load a test file
    transfer = AcousticTransfer()
    test_audio, sr = transfer.load_audio('content_audio/test_0.wav')
    
    # Get all environments
    environments = list(Path('impulse_responses').glob('*.wav'))
    
    # Create visualization
    plt.figure(figsize=(15, 10))
    
    for i, env_file in enumerate(environments):
        env_name = env_file.stem
        
        # Load impulse response
        impulse, _ = transfer.load_audio(str(env_file))
        
        # Apply transformation
        transformed = transfer.apply_convolution_reverb(test_audio, impulse)
        
        # Plot results
        plt.subplot(3, 2, i+1)
        
        # Plot impulse response
        time_ir = np.linspace(0, len(impulse)/sr, len(impulse))
        plt.plot(time_ir, impulse, alpha=0.7, label='Impulse Response')
        
        # Plot transformed audio (first 0.5 seconds)
        time_audio = np.linspace(0, 0.5, min(len(transformed), int(0.5*sr)))
        audio_sample = transformed[:len(time_audio)]
        plt.plot(time_audio, audio_sample * 0.5 + 0.3, alpha=0.7, label='Transformed Audio')
        
        plt.title(f'{env_name.replace("_", " ").title()}')
        plt.xlabel('Time (s)')
        plt.ylabel('Amplitude')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # Save audio file
        output_file = f'DEMO_{env_name}.wav'
        sf.write(output_file, transformed, sr)
        print(f"✅ Created: {output_file}")
    
    plt.tight_layout()
    plt.savefig('acoustic_transfer_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n🎉 DEMO COMPLETE!")
    print("Generated files:")
    for env_file in environments:
        env_name = env_file.stem
        print(f"  • DEMO_{env_name}.wav - Audio in {env_name} environment")
    print("  • acoustic_transfer_results.png - Visualization")
    
    # Play the first result
    from IPython.display import Audio, display
    print("\n🔊 Playing original audio:")
    display(Audio(test_audio, rate=sr))
    
    print("🔊 Playing transformed audio (small room):")
    transformed_audio, _ = transfer.load_audio('DEMO_small_room.wav')
    display(Audio(transformed_audio, rate=sr))

# Run the complete demo
run_complete_demo()

In [12]:
# Simple function to transform any audio file
def transform_audio_to_environment(input_audio_path, environment_name, output_path=None):
    """
    Transform audio to sound like a specific environment
    
    Args:
        input_audio_path: Path to input audio file
        environment_name: One of ['small_room', 'large_hall', 'church', 'stairwell']
        output_path: Optional output path (default: auto-generated)
    """
    transfer = AcousticTransfer()
    
    # Load audio
    dry_audio, sr = transfer.load_audio(input_audio_path)
    
    # Load impulse response
    impulse_path = f'impulse_responses/{environment_name}.wav'
    impulse, _ = transfer.load_audio(impulse_path)
    
    # Apply transformation
    transformed_audio = transfer.apply_convolution_reverb(dry_audio, impulse)
    
    # Save result
    if output_path is None:
        output_path = f'transformed_{environment_name}.wav'
    
    sf.write(output_path, transformed_audio, sr)
    
    print(f"✅ Transformed {input_audio_path} to {environment_name} environment")
    print(f"✅ Saved as: {output_path}")
    
    return transformed_audio

# Example usage:
# transformed = transform_audio_to_environment('my_voice.wav', 'large_hall')